In [1]:
# ================================

# --- 0) Install deps ---
!pip -q install timm==1.0.9

import os, zipfile, shutil, random, json, time
from pathlib import Path

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, random_split
from torchvision import datasets, transforms

import timm
import numpy as np
from PIL import Image
from sklearn.metrics import classification_report, confusion_matrix
import itertools

# ============ USER CONFIG ============
# 1) Point this to your ZIP inside /kaggle/input
ZIP_PATH = "/kaggle/input/broadridge2-0"  

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.4/42.4 kB 2.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 48.6 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 92.4 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 71.2 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 37.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 7.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 15.8 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 13.7 MB/s eta 0:00:0000:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 7.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [5]:
import os

DATASET_PATH = "/kaggle/input/broadridge2-0"   # This is already a folder Kaggle extracted

# Look at top-level folders
for root, dirs, files in os.walk(DATASET_PATH):
    if dirs:
        print("Folder:", root, " -> Subfolders:", dirs)
    if files and any(f.lower().endswith(('.jpg','.jpeg','.png')) for f in files):
        print("Images inside:", root, "| Count:", len(files))


Folder: /kaggle/input/broadridge2-0  -> Subfolders: ['Stocks', 'ETFs', 'Data']
Folder: /kaggle/input/broadridge2-0/Data  -> Subfolders: ['Stocks', 'ETFs']


In [6]:
import os

base_path = "/kaggle/input/broadridge2-0"

for folder in os.listdir(base_path):
    folder_path = os.path.join(base_path, folder)
    if os.path.isdir(folder_path):
        print(f"\n📂 {folder}")
        files = os.listdir(folder_path)[:10]  # just first 10 files
        print("   Sample files:", files)



📂 Stocks
   Sample files: ['ufi.us.txt', 'vfl.us.txt', 'sohu.us.txt', 'rdcm.us.txt', 'virt.us.txt', 'hylb.us.txt', 'skt.us.txt', 'asix.us.txt', 'mac.us.txt', 'gut.us.txt']

📂 ETFs
   Sample files: ['djci.us.txt', 'sqqq.us.txt', 'ipac.us.txt', 'vb.us.txt', 'cper.us.txt', 'rjz.us.txt', 'pdn.us.txt', 'ixn.us.txt', 'acwi.us.txt', 'pscc.us.txt']

📂 Data
   Sample files: ['Stocks', 'ETFs']


In [7]:
import os
import pandas as pd

# Base path
base_path = "/kaggle/input/broadridge2-0"

# Function to load and preprocess a folder (Stocks or ETFs)
def load_data(folder):
    folder_path = os.path.join(base_path, folder)
    all_data = []

    for file in os.listdir(folder_path):
        if file.endswith(".txt"):
            file_path = os.path.join(folder_path, file)
            df = pd.read_csv(file_path)
            
            # Add metadata
            ticker = file.replace(".us.txt", "")
            df["Ticker"] = ticker

            # Standardize column names if needed
            df.columns = [col.strip().capitalize() for col in df.columns]

            # Convert Date column
            if "Date" in df.columns:
                df["Date"] = pd.to_datetime(df["Date"], errors="coerce")
            
            all_data.append(df)

    # Merge all data
    return pd.concat(all_data, ignore_index=True)

# Load stocks and ETFs
stocks_df = load_data("Stocks")
etfs_df = load_data("ETFs")

print("Stocks shape:", stocks_df.shape)
print("ETFs shape:", etfs_df.shape)

# Quick look
print(stocks_df.head())


EmptyDataError: No columns to parse from file

In [8]:
import os
import pandas as pd

# Base path
base_path = "/kaggle/input/broadridge2-0"

def load_data(folder):
    folder_path = os.path.join(base_path, folder)
    all_data = []

    for file in os.listdir(folder_path):
        if file.endswith(".txt"):
            file_path = os.path.join(folder_path, file)

            # Skip empty files
            if os.path.getsize(file_path) == 0:
                print(f"⚠️ Skipping empty file: {file}")
                continue

            try:
                df = pd.read_csv(file_path)

                # Add ticker column
                ticker = file.replace(".us.txt", "")
                df["Ticker"] = ticker

                # Standardize column names
                df.columns = [col.strip().capitalize() for col in df.columns]

                # Convert Date column
                if "Date" in df.columns:
                    df["Date"] = pd.to_datetime(df["Date"], errors="coerce")

                all_data.append(df)
            
            except Exception as e:
                print(f"⚠️ Skipping {file} due to error: {e}")

    # Merge all valid dataframes
    if all_data:
        return pd.concat(all_data, ignore_index=True)
    else:
        return pd.DataFrame()

# Load stocks and ETFs
stocks_df = load_data("Stocks")
etfs_df = load_data("ETFs")

print("Stocks shape:", stocks_df.shape)
print("ETFs shape:", etfs_df.shape)
print(stocks_df.head())


⚠️ Skipping empty file: sbt.us.txt
⚠️ Skipping empty file: srva.us.txt
⚠️ Skipping empty file: pxus.us.txt
⚠️ Skipping empty file: stnl.us.txt
⚠️ Skipping empty file: bolt.us.txt
⚠️ Skipping empty file: send.us.txt
⚠️ Skipping empty file: bbrx.us.txt
⚠️ Skipping empty file: scci.us.txt
⚠️ Skipping empty file: scph.us.txt
⚠️ Skipping empty file: gnst.us.txt
⚠️ Skipping empty file: wspt.us.txt
⚠️ Skipping empty file: boxl.us.txt
⚠️ Skipping empty file: vist.us.txt
⚠️ Skipping empty file: fmax.us.txt
⚠️ Skipping empty file: sfix.us.txt
⚠️ Skipping empty file: amrhw.us.txt
⚠️ Skipping empty file: jt.us.txt
⚠️ Skipping empty file: vmet.us.txt
⚠️ Skipping empty file: accp.us.txt
⚠️ Skipping empty file: pbio.us.txt
⚠️ Skipping empty file: wnfm.us.txt
⚠️ Skipping empty file: hayu.us.txt
⚠️ Skipping empty file: amrh.us.txt
⚠️ Skipping empty file: sail.us.txt
⚠️ Skipping empty file: molc.us.txt
⚠️ Skipping empty file: otg.us.txt
⚠️ Skipping empty file: znwaa.us.txt
⚠️ Skipping empty file: rbio.u